# gnn_data

In [1]:
import os
import numpy as np
import pandas as pd
from biopandas.pdb import PandasPdb
from biopandas.mmcif import PandasMmcif
import torch
import dgl
import tokenizers
import transformers
print(f"tokenizers.__version__: {tokenizers.__version__}")
print(f"transformers.__version__: {transformers.__version__}")
from transformers import AutoTokenizer, AutoModel, AutoConfig

tokenizers.__version__: 0.19.1
transformers.__version__: 4.44.2


In [2]:
def get_distance_matrix(coords):
    diff_tensor = np.expand_dims(coords, axis=1) - np.expand_dims(coords, axis=0)
    distance_matrix = np.sqrt(np.sum(np.power(diff_tensor, 2), axis=-1))
    return distance_matrix

def generate_graph(pdb_path, distance_threshold=8.0):
    atom_df = PandasPdb().read_pdb(pdb_path)
    atom_df = atom_df.df['ATOM']
    residue_df = atom_df.groupby('residue_number', as_index=False)[['x_coord', 'y_coord', 'z_coord', 'b_factor']].mean().sort_values('residue_number')
    coords = residue_df[['x_coord', 'y_coord', 'z_coord']].values
    distance_matrix = get_distance_matrix(coords)
    adj = distance_matrix < distance_threshold
    u, v = np.nonzero(adj)
    u, v = torch.from_numpy(u), torch.from_numpy(v)
    graph = dgl.graph((u, v), num_nodes=len(coords))
    return graph

In [3]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, alphabet = torch.hub.load("facebookresearch/esm:main", "esm2_t33_650M_UR50D")
batch_converter = alphabet.get_batch_converter()
model.to(device)
model.eval()  # disables dropout for deterministic results
def esm_encode(seq):
    data = [("protein", seq)]
    batch_labels, batch_strs, batch_tokens = batch_converter(data)
    batch_tokens = batch_tokens.to(device)
    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[33], return_contacts=True)
    token_representations = results["representations"][33]
    sequence_representation = token_representations.squeeze()[1:-1, :].cpu()
    return sequence_representation 

Using cache found in /home/hongchang/.cache/torch/hub/facebookresearch_esm_main


In [4]:
aa_map = {'VAL': 'V', 'PRO': 'P', 'ASN': 'N', 'GLU': 'E', 'ASP': 'D', 'ALA': 'A', 'THR': 'T', 'SER': 'S',
          'LEU': 'L', 'LYS': 'K', 'GLY': 'G', 'GLN': 'Q', 'ILE': 'I', 'PHE': 'F', 'CYS': 'C', 'TRP': 'W',
          'ARG': 'R', 'TYR': 'Y', 'HIS': 'H', 'MET': 'M'}
aa_props = pd.read_csv('aminoacids.csv').set_index('Letter')
aa_props2 = aa_props[['Molecular Weight', 'Residue Weight', 'pKa1', 'pKb2', 'pl4', 
         'H', 'VSC', 'P1', 'P2', 'SASA', 'NCISC', 'carbon', 'hydrogen', 'nitrogen',
       'oxygen', 'sulfur']]
propp = ['pKa1', 'pKb2', 'pl4', 'H', 'VSC', 'P1', 'P2', 'SASA', 'NCISC', 'carbon', 'hydrogen', 'nitrogen', 'oxygen', 'sulfur']
aa_props2 = aa_props2.fillna(aa_props2.mean())
min_weight = np.min(aa_props2[['Molecular Weight', 'Residue Weight']].values)
max_weight = np.max(aa_props2[['Molecular Weight', 'Residue Weight']].values)
aa_props2[['Molecular Weight', 'Residue Weight']] = (aa_props2[['Molecular Weight', 'Residue Weight']] - min_weight) / (max_weight - min_weight)
aa_props2[propp] = (aa_props2[propp]-aa_props2[propp].min()) / (aa_props2[propp].max()-aa_props2[propp].min())
def physchem_encode(seq):
    letter_index = list(seq)
    encoded_seq = aa_props2.loc[letter_index].values
    return encoded_seq.astype(np.float32)

# predict

In [5]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from scipy.stats import rankdata
import torch
import dgl
import time
from tqdm import tqdm
import random
import gc
import dgl.function as fn
from dgl.nn import GATConv

In [6]:
class GraphDataset(torch.utils.data.Dataset):
    def __init__(self, dir_path, indexes=None, add_self_loop=False):
        super(GraphDataset, self).__init__()
        self.dir_path = dir_path
        self.graphs, label_dict = dgl.load_graphs(self.dir_path+'/dgl_graph.bin')
        self.df = pd.read_csv(self.dir_path+'/overview_df.csv', index_col=0)
        self.add_self_loop = add_self_loop
        if indexes is None:
            self.indexes = self.df.index
        else:
            self.indexes = indexes

    def __getitem__(self, i):
        idx = self.indexes[i]
        
        row = self.df.loc[idx]
        
        graph_index = row.graph_index
        graph = self.graphs[graph_index].clone()
        if self.add_self_loop:
            graph = dgl.add_self_loop(graph)
        
        seq_feature = np.load(self.dir_path+'/'+row.seq_feature_path)
        seq = seq_feature['seq']
        seq = torch.tensor(seq) 
        seq = torch.cat((seq[:, :1280], seq[:, -16:]), dim=1)
        
        surface_aa_feature = np.load(self.dir_path+'/'+row.surface_aa_feature_path)
        surface_aa_seq = surface_aa_feature['surface_aa_seq']
        surface_aa_seq = torch.tensor(surface_aa_seq) 
        surface_aa_seq = torch.cat((surface_aa_seq[:, :1280], surface_aa_seq[:, -16:]), dim=1)
        
        surface_pos = surface_aa_feature['surface_pos']
        
        graph.ndata['seq'] = seq
        graph.ndata['surface_aa_seq'] = surface_aa_seq
        graph.ndata['surface_pos'] = torch.from_numpy(surface_pos)
        
        label = row.get('pHmin', np.nan)
        label_valid = True
             
        return graph, label, label_valid, idx

    def __len__(self):
        return len(self.indexes)

In [7]:
import torch
import torch.nn as nn
import dgl
from dgl.nn.pytorch import GraphConv

class GCN(nn.Module):
    def __init__(self, hidden_dim, layer_num=3):
        super(GCN, self).__init__()
        self.convs = nn.ModuleList()
        self.activations = nn.ModuleList()
        self.batch_norms = nn.ModuleList()

        for i in range(layer_num):
            in_feats = hidden_dim if i == 0 else hidden_dim
            out_feats = hidden_dim
            self.convs.append(GraphConv(in_feats, out_feats))
            self.activations.append(nn.LeakyReLU())
            self.batch_norms.append(nn.BatchNorm1d(out_feats))

        self.layer_num = layer_num
        self.out_dim = hidden_dim * layer_num
        
    def forward(self, g, h):
        hs = [h]
        for conv, batch_norm, act in zip(self.convs, self.batch_norms, self.activations):
            h = conv(g, h)
            h = batch_norm(h)
            h = act(h)
            hs.append(h)
        return torch.cat(hs, dim=-1)
class GNNModel(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim=256, dropout_rate=0.5):
        super(GNNModel, self).__init__()
        self.comp = torch.nn.Sequential(
            torch.nn.Linear(in_dim, hidden_dim),
            torch.nn.LeakyReLU()
        )
        self.gcn = GCN(hidden_dim)
        self.head = torch.nn.Sequential(
            torch.nn.Dropout(0.5),
            torch.nn.Linear(2048, self.gcn.out_dim),
            torch.nn.LeakyReLU(),
            torch.nn.Dropout(0.5),
            torch.nn.Linear(self.gcn.out_dim, 1),
        )
    
    def forward(self, g, wildtype_seq, surface_aa_seq, surface_pos):
        wildtype_h = self.comp(wildtype_seq)
        surface_aa_h = self.comp(surface_aa_seq)
        wildtype_h = self.gcn(g, wildtype_h)
        surface_aa_h = self.gcn(g, surface_aa_h)
        with g.local_scope():
            g.ndata['h'] = wildtype_h
            wildtype_hg = dgl.readout_nodes(g, 'h', op='sum') 

        with g.local_scope():
            g.ndata['h'] = surface_aa_h
            surface_aa_hp = dgl.readout_nodes(g, 'h', op='sum')
        h_all = torch.cat([wildtype_hg, surface_aa_hp], dim=-1)
        pred = self.head(h_all).squeeze()
        return pred

In [8]:
import ast
import os
import pickle
from scipy.stats import pearsonr

JJRMFPAS = GNNModel(1296, 256).cuda()
JJRMFPAS.load_state_dict(torch.load('ACENet.pth'))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
JJRMFPAS.eval()
test_dataset = GraphDataset('EGFP')
test_dataloader = dgl.dataloading.GraphDataLoader(test_dataset, batch_size=1, shuffle=False, drop_last=False, num_workers=26)
graph, label, label_valid, original_index = next(iter(test_dataset))
test_predictions = []
test_pos = []
graph = graph.to(device)
seq, surface_aa_seq, surface_pos = graph.ndata['seq'], graph.ndata['surface_aa_seq'], graph.ndata['surface_pos']
df = pd.read_csv('EGFP.csv')  

with torch.no_grad():   
    pred_0 = JJRMFPAS(graph, seq, surface_aa_seq, surface_pos).item()
print(pred_0)
for pos in eval(df['surface_index'][0]):
    pos = int(pos)
    seq_1 = seq.clone()
    seq_2 = surface_aa_seq.clone()
    seq_1[pos-1, :] = 0
    seq_2[pos-1, :] = 0
    graph_0 = graph.clone()
    with torch.no_grad():   
        pred = JJRMFPAS(graph_0, seq_1, seq_2, surface_pos)
    test_predictions.append(pred.item()-pred_0)
    test_pos.append(pos)

df = pd.DataFrame()
df['test_predictions'] = test_predictions
df['pos'] = test_pos
#df.to_csv(f'result/{path}.csv', index=False)

/tmp/ipykernel_146142/3598041109.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  JJRMFPAS.load_state_dict(torch.load('JJ-RMFPAS.pth'))


5.474311351776123


In [9]:
print(len(seq))

239


In [10]:
import pandas as pd
df['abs_test_predictions'] = df['test_predictions'].abs()
top_30 = df.nlargest(50, 'abs_test_predictions') 
top_30 = top_30.iloc[1:]

In [11]:
from Bio import SeqIO
def get_protein_sequence_from_pdb(pdb_path):
    with open(pdb_path, "r") as pdb_file:
        for record in SeqIO.parse(pdb_file, "pdb-atom"):
            return str(record.seq)

In [12]:
df = pd.read_csv('EGFP.csv')
surface_index = df['surface_index'][0]
position =eval(surface_index)

In [13]:
#第一轮设计
JJRMFPAS = GNNModel(1296, 256).cuda()
JJRMFPAS.load_state_dict(torch.load('JJ-RMFPAS.pth'))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
JJRMFPAS.eval()
surface_pos = np.zeros(graph.num_nodes())
test_predictions_1 = []
test_pos_1 = []
with torch.no_grad():   
    pred_0 = JJRMFPAS(graph, seq, surface_aa_seq, surface_pos).item()
import warnings
warnings.filterwarnings('ignore')
for index,row in tqdm(top_30.iterrows(), total=len(top_30)): 
    pos = int(row['pos'])
    sequence = get_protein_sequence_from_pdb(f"pdb/EGFP.pdb")
    for aa in ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']:
        mut_sequence = sequence[:pos-1] + aa + sequence[pos:]
        esm_encoded_seq = esm_encode(mut_sequence).cpu().numpy()
        physchem_encoded_seq = physchem_encode(mut_sequence)
        encoded_seq = np.concatenate([esm_encoded_seq , physchem_encoded_seq], axis=-1)
        encoded_seq = torch.tensor(encoded_seq).to(device)
        for pdb_position_index in position:
            surface_pos[pdb_position_index-1] = 1
        surface_pos = torch.tensor(surface_pos)

        encoded_surface = encoded_seq.clone()
        encoded_surface[surface_pos == 0] = 0
        encoded_surface = torch.tensor(encoded_surface).to(device)
        with torch.no_grad():   
            pred = JJRMFPAS(graph, encoded_seq, encoded_surface, surface_pos)
        
        test_predictions_1.append(pred_0-pred.item())
        index = str(pos-1)
        test_pos_1.append(index+aa)

/tmp/ipykernel_146142/2608430319.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  JJRMFPAS.load_state_dict(torch.load('JJ-RMFPAS.pth'))
100%|█████████████████████████████

In [14]:
print(test_predictions)
print(test_pos)

[-0.018686294555664062, -0.042652130126953125, -0.02181720733642578, -0.022445201873779297, -0.008177757263183594, -0.016832351684570312, -0.01892852783203125, -0.007291316986083984, -0.04700469970703125, 0.005198478698730469, -0.006109714508056641, -0.04371452331542969, -0.03583717346191406, 0.0027604103088378906, -0.008536815643310547, -0.01834249496459961, -0.027794361114501953, -0.015415191650390625, -0.023315906524658203, -0.01675891876220703, -0.01044464111328125, -0.012698173522949219, 0.003627300262451172, -0.004811286926269531, -0.013285636901855469, 0.0026884078979492188, -0.11310720443725586, -0.039999961853027344, -0.0250701904296875, -0.012021064758300781, 0.0017743110656738281, -0.06294822692871094, -0.006516456604003906, -0.02548360824584961, -0.013389110565185547, -0.005074501037597656, -0.030772686004638672, -0.03017902374267578, -0.008838653564453125, 0.004345417022705078, -0.010945796966552734, -0.0056705474853515625, -0.06013345718383789, 0.0032396316528320312, -0.0

In [15]:
def get_top_50_indices(numbers):
    if len(numbers) < 50:
        raise ValueError("List must contain at least 50 elements.")
    
    sorted_numbers_with_index = sorted(enumerate(numbers), key=lambda x: x[1], reverse=True)
    top_50_indices = [index for index, value in sorted_numbers_with_index[:50]]
    
    return top_50_indices 

def get_values_from_indices(top_indices, values_list):
    top_values = [values_list[index] for index in top_indices]
    return top_values

top_indices_1 = get_top_50_indices(test_predictions_1)
top_mut_1 = get_values_from_indices(top_indices_1, test_pos_1)
print(top_mut_1)
pred_top_mut_1 = get_values_from_indices(top_indices_1, test_predictions_1)
print(pred_top_mut_1)

['42G', '44S', '44N', '165R', '44P', '34M', '44Q', '51Y', '42A', '44D', '42S', '32F', '44R', '34F', '165Q', '32R', '44H', '32M', '32L', '225G', '44E', '34I', '224P', '32Y', '51W', '32W', '181A', '224R', '31N', '165K', '181M', '51M', '44G', '17P', '34A', '48R', '34L', '34V', '165D', '224F', '34H', '224N', '144L', '32H', '32K', '144M', '44T', '230W', '165E', '230G']
[0.5179924964904785, 0.5067138671875, 0.5026555061340332, 0.49048519134521484, 0.46805238723754883, 0.46778106689453125, 0.44694948196411133, 0.4386320114135742, 0.4273643493652344, 0.4196653366088867, 0.41557788848876953, 0.4113454818725586, 0.4085111618041992, 0.40758514404296875, 0.4044308662414551, 0.3913888931274414, 0.3828582763671875, 0.38210105895996094, 0.3783726692199707, 0.3691253662109375, 0.36514711380004883, 0.36474084854125977, 0.3587765693664551, 0.35491371154785156, 0.35182857513427734, 0.35019683837890625, 0.34331321716308594, 0.3383927345275879, 0.3380250930786133, 0.33776378631591797, 0.33698606491088867, 

In [16]:
#第二轮设计
import itertools
test_predictions_2 = []
test_pos_2 = []
for combo in tqdm(itertools.combinations(top_mut_1, 2), total=len(list(itertools.combinations(top_mut_1, 2)))):
    mut_1, mut_2 = combo
    mut_1_pos = int(mut_1[:-1])  
    mut_1_aa = mut_1[-1]   
    mut_2_pos = int(mut_2[:-1])  
    mut_2_aa = mut_2[-1]  
    if mut_1_pos == mut_2_pos:
        continue
    else:
        sequence = get_protein_sequence_from_pdb(f"pdb/EGFP.pdb")
        mut_sequence = sequence[:mut_1_pos] + mut_1_aa + sequence[mut_1_pos+1:]
        mut_sequence = mut_sequence[:mut_2_pos] + mut_2_aa + mut_sequence[mut_2_pos+1:]
        esm_encoded_seq = esm_encode(mut_sequence).cpu().numpy()
        physchem_encoded_seq = physchem_encode(mut_sequence)
        encoded_seq = np.concatenate([esm_encoded_seq , physchem_encoded_seq], axis=-1)
        encoded_seq = torch.tensor(encoded_seq).to(device)
        for pdb_position_index in position:
            surface_pos[pdb_position_index-1] = 1
        surface_pos = torch.tensor(surface_pos)

        encoded_surface = encoded_seq.clone()
        encoded_surface[surface_pos == 0] = 0
        encoded_surface = torch.tensor(encoded_surface).to(device)
        with torch.no_grad():   
            pred = JJRMFPAS(graph, encoded_seq, encoded_surface, surface_pos)
        
        test_predictions_2.append(pred_0-pred.item())
        index_1 = str(mut_1_pos)
        index_2 = str(mut_2_pos)
        test_pos_2.append(index_1+mut_1_aa+';'+index_2+mut_2_aa)

100%|███████████████████████████████████████| 1225/1225 [01:31<00:00, 13.38it/s]


In [17]:
top_indices_2 = get_top_50_indices(test_predictions_2)
top_mut_2 = get_values_from_indices(top_indices_2, test_pos_2)
print(top_mut_2)
pred_top_mut_2 = get_values_from_indices(top_indices_2, test_predictions_2)

['44S;165R', '44N;224P', '44S;165Q', '44N;224R', '44H;224P', '44D;224P', '44Q;224P', '51Y;225G', '44S;181A', '44N;224N', '42G;34A', '165R;44Q', '44N;34A', '44N;225G', '44S;181M', '44S;225G', '42G;225G', '34M;181M', '44H;224R', '165R;42S', '44S;34A', '44Q;225G', '42G;224P', '44R;225G', '42G;224R', '165R;44T', '44E;224P', '44P;224P', '44S;224F', '44R;224P', '34M;181A', '224P;34A', '42A;224F', '44G;34A', '165R;42A', '165R;225G', '225G;51M', '44Q;34A', '31N;230G', '42G;48R', '44Q;224R', '42G;165R', '44Q;181M', '165R;44H', '34A;230W', '42A;51W', '42G;181A', '44H;225G', '42A;34A', '42S;224F']


In [18]:
pred_top_mut_2 

[0.9836931228637695,
 0.9799494743347168,
 0.9484400749206543,
 0.9131875038146973,
 0.8927159309387207,
 0.8608651161193848,
 0.8603167533874512,
 0.8555021286010742,
 0.8534383773803711,
 0.8451452255249023,
 0.8440790176391602,
 0.8401265144348145,
 0.8240985870361328,
 0.8228192329406738,
 0.822418212890625,
 0.8198633193969727,
 0.8198103904724121,
 0.8188085556030273,
 0.817840576171875,
 0.8171381950378418,
 0.8144736289978027,
 0.8117809295654297,
 0.8115987777709961,
 0.8109378814697266,
 0.809535026550293,
 0.8061132431030273,
 0.8051056861877441,
 0.8044123649597168,
 0.8030214309692383,
 0.8020782470703125,
 0.7963147163391113,
 0.7963047027587891,
 0.7959351539611816,
 0.7928953170776367,
 0.7928290367126465,
 0.7885255813598633,
 0.7883844375610352,
 0.7862834930419922,
 0.7834100723266602,
 0.7827987670898438,
 0.7818470001220703,
 0.7804150581359863,
 0.7797322273254395,
 0.7795190811157227,
 0.779296875,
 0.7791261672973633,
 0.7770156860351562,
 0.7763676643371582,
 0

In [19]:
import re

def extract_parts(input_string):
    pattern = r'(\d+)(\D+);(\d+)(\D+)'
    match = re.match(pattern, input_string)
    if match:
        number1 = match.group(1)  
        letter1 = match.group(2)  
        number2 = match.group(3)  
        letter2 = match.group(4)  
        
        return number1, letter1, number2, letter2
    else:
        raise ValueError("not match")

In [20]:
test_predictions_3 = []
test_pos_3 = []
for Mut_2 in tqdm(top_mut_2, total = len(top_mut_2)):
    mut_1_pos, mut_1_aa, mut_2_pos, mut_2_aa = extract_parts(Mut_2)
    mut_1_pos = int(mut_1_pos)
    mut_2_pos = int(mut_2_pos)
    sequence = get_protein_sequence_from_pdb(f"pdb/EGFP.pdb")
    mut_sequence = sequence[:mut_1_pos] + mut_1_aa + sequence[mut_1_pos+1:]
    mut_sequence = mut_sequence[:mut_2_pos] + mut_2_aa + mut_sequence[mut_2_pos+1:]
    for Mut_3 in top_mut_1:
        mut_3_pos = int(Mut_3[:-1])  
        mut_3_aa = Mut_3[-1]
        if mut_3_pos == mut_1_pos or mut_3_pos == mut_2_pos:
            continue
        else:
            mut_sequence = mut_sequence[:mut_3_pos] + mut_3_aa + mut_sequence[mut_3_pos+1:]
            esm_encoded_seq = esm_encode(mut_sequence).cpu().numpy()
            physchem_encoded_seq = physchem_encode(mut_sequence)
            encoded_seq = np.concatenate([esm_encoded_seq , physchem_encoded_seq], axis=-1)
            encoded_seq = torch.tensor(encoded_seq).to(device)
            for pdb_position_index in position:
                surface_pos[pdb_position_index-1] = 1
            surface_pos = torch.tensor(surface_pos)
    
            encoded_surface = encoded_seq.clone()
            encoded_surface[surface_pos == 0] = 0
            encoded_surface = torch.tensor(encoded_surface).to(device)
            with torch.no_grad():   
                pred = JJRMFPAS(graph, encoded_seq, encoded_surface, surface_pos)
            
            test_predictions_3.append(pred_0-pred.item())
            index_1 = str(mut_1_pos)
            index_2 = str(mut_2_pos)
            index_3 = str(mut_3_pos)
            test_pos_3.append(index_1+mut_1_aa+';'+index_2+mut_2_aa+';'+index_3+mut_3_aa)

100%|███████████████████████████████████████████| 50/50 [01:44<00:00,  2.09s/it]


In [21]:
top_indices_3 = get_top_50_indices(test_predictions_3)
top_mut_3 = get_values_from_indices(top_indices_3, test_pos_3)
print(top_mut_3)
pred_top_mut_3 = get_values_from_indices(top_indices_3, test_predictions_3)

['44Q;34A;224N', '44S;34A;224N', '42G;181A;48R', '44N;224N;34H', '44N;225G;224N', '44Q;34A;165D', '44N;224N;165D', '44Q;225G;224N', '44Q;181M;224N', '34A;230W;224N', '44N;34A;165D', '44Q;225G;144L', '44Q;181M;144L', '44S;181A;224N', '42G;34A;224F', '44S;34A;165D', '34A;230W;165D', '42G;34A;48R', '42G;225G;48R', '42G;224R;48R', '42G;48R;34A', '44Q;34A;144L', '42G;34A;165E', '34M;181A;224F', '42G;34A;230G', '44S;181M;224N', '44S;225G;224N', '44S;165Q;144L', '42G;181A;34L', '42G;34A;144M', '44N;34A;224N', '44S;181M;144L', '44S;225G;144L', '42A;34A;224F', '31N;230G;224N', '42G;34A;32K', '44N;34A;224F', '225G;51M;224N', '34A;230W;48R', '42G;181A;224F', '44N;224R;165D', '44N;225G;165D', '44N;224N;48R', '225G;51M;165D', '44G;34A;224F', '44S;34A;144L', '42G;34A;44T', '44N;224R;34H', '42G;181A;34V', '44Q;34A;224F']


In [22]:
pred_top_mut_3 

[1.7647955417633057,
 1.719496488571167,
 1.7003531455993652,
 1.6831881999969482,
 1.6831881999969482,
 1.6812138557434082,
 1.6756877899169922,
 1.6739513874053955,
 1.6739513874053955,
 1.670295238494873,
 1.6687443256378174,
 1.666914939880371,
 1.666914939880371,
 1.6621315479278564,
 1.6599407196044922,
 1.6442465782165527,
 1.6420488357543945,
 1.6405656337738037,
 1.6405656337738037,
 1.6405656337738037,
 1.6405656337738037,
 1.6390676498413086,
 1.6369104385375977,
 1.6356885433197021,
 1.6230790615081787,
 1.6207239627838135,
 1.6207239627838135,
 1.6161746978759766,
 1.6153903007507324,
 1.6144216060638428,
 1.6136937141418457,
 1.6134328842163086,
 1.6134328842163086,
 1.6116528511047363,
 1.6107151508331299,
 1.6096723079681396,
 1.6092236042022705,
 1.60556960105896,
 1.6028048992156982,
 1.6024730205535889,
 1.598780632019043,
 1.598780632019043,
 1.5975096225738525,
 1.596574068069458,
 1.5935022830963135,
 1.5927433967590332,
 1.592430591583252,
 1.589878797531128,
 1.

In [23]:
import pandas as pd
df = pd.DataFrame({
    'Prediction': pred_top_mut_3,
    'Actual': top_mut_3
})
df.to_csv('mut_EGFP.csv', index=False)